# The field-realistic section: geology in, seismic out

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mosesfalowo/ogbonlab-notebooks/blob/main/notebooks/ssm_field_realistic_section.ipynb)


Generated by [OgbonLab](https://ogbonlab.com), Synthetic Seismic Modeling course.

**Assumptions**: The processed, migrated-image idealization: per-trace 1D convolution in time of a geologically built earth. Edit the parameters cell and run again.

**Workflow**:
1. Fold, fault, and truncate a deep package of thin beds
2. Drape a post-unconformity package with a channel over it
3. Assign properties from a compaction trend and Gardner density
4. Convert depth to time per trace and image with a time-variant wavelet
5. Add band-limited noise at survey-quality S/N

In [ ]:
# The field-realistic section: geology in, seismic out
# Generated by OgbonLab (ogbonlab.com), Synthetic Seismic Modeling course.
import numpy as np
import matplotlib.pyplot as plt

# --- parameters ---
SEED = 7          # change it: a different random overprint, same geology
SNR = 5.0         # signal-to-noise of the final image
DT_S = 0.002      # trace sample interval (s)

In [ ]:
# --- the Ogbon standard section: a field-realistic 2D earth ---------
# Geology in geological order: a deep package of thin parallel beds
# (regional dip + an asymmetric anticline) cut by three normal faults;
# an angular unconformity that peneplanes the fold crest, so the folded
# beds terminate against it and one soft reservoir sand subcrops right
# at the crest (a subcrop trap); a draped post-unconformity package
# with a channel incised into one horizon. Properties come from a
# compaction trend with sand/shale alternation and Gardner density.

def ricker(f, dt, n):
    t = (np.arange(n) - n//2) * dt
    a = (np.pi * f * t) ** 2
    return (1 - 2*a) * np.exp(-a)

def build_field_model(nx=360, nz=300, dx=12.5, dz=5.0, seed=7):
    rng = np.random.default_rng(seed)
    x = np.arange(nx) * dx
    xn = x / x[-1]
    zgrid = np.arange(nz) * dz
    # the fold: asymmetric anticline, steep left flank
    sig = np.where(xn < 0.44, 0.105, 0.21)
    fold = np.exp(-0.5 * ((xn - 0.44) / sig) ** 2)
    # deep package: thin parallel beds, dip + fold
    n_deep = 23
    sp = 42.0 + 10.0 * rng.random(n_deep)
    bases = 470.0 + np.cumsum(sp) - sp[0]
    deep = np.empty((n_deep, nx))
    for k in range(n_deep):
        tv = 1.0 + 0.035 * np.sin(2*np.pi * (xn*0.7 + 0.29*k))
        deep[k] = bases[k] * tv + 225.0 * xn - 235.0 * fold
    # normal faults: throw grows with depth, dies out upward
    faults = [(0.615, -0.30, 95.0, True, 430.0),
              (0.760, +0.38, 52.0, False, 470.0),
              (0.230, -0.24, 36.0, False, 560.0)]
    def fault_x(f, z):
        x0, sl, _, listric, _ = f
        s = sl * (1.0 + (0.55 * z/1500.0 if listric else 0.0))
        return x0 + s * (z - 900.0) / 1400.0
    def fault_throw(f, z):
        return f[2] * np.clip((z - f[4]) / (1150.0 - f[4]), 0.0, 1.35)
    for f in faults:
        for k in range(n_deep):
            hw = 1.0 / (1.0 + np.exp(-(xn - fault_x(f, deep[k])) / 0.0035))
            deep[k] = deep[k] + fault_throw(f, deep[k]) * hw
    # angular unconformity truncates the folded beds
    unconf = 415.0 + 105.0*xn + 9.0*np.sin(2*np.pi*(xn*1.3 + 0.2))
    deep = np.maximum(deep, unconf)
    # shallow package: drapes the unconformity, flattens upward,
    # gentle differential compaction over the buried crest
    n_sh = 8
    u_rel = unconf - unconf.mean()
    sh = np.empty((n_sh, nx))
    for k in range(n_sh):
        fr = k / (n_sh - 1.0)
        sh[k] = (55.0 + (unconf.min() - 95.0) * fr
                 + (0.10 + 0.80 * fr**1.7) * u_rel
                 - (4.0 + 26.0 * fr) * (fold - fold.mean()))
    ch_k = 2   # channel incised into the third shallow horizon
    ch = sh[ch_k] + 55.0 * np.exp(-0.5 * ((xn - 0.66) / 0.05) ** 2)
    tops = np.vstack([sh, ch[None, :], deep])
    tops = np.maximum.accumulate(np.sort(tops, axis=0), axis=0)
    # properties: compaction trend, alternation, markers, Gardner
    n_lay = tops.shape[0] + 1
    mids = np.r_[28.0, 0.5*(tops[:-1, 0] + tops[1:, 0]), tops[-1, 0] + 35.0]
    vp = (1720.0 + 1.05 * mids) \
        * np.where(np.arange(n_lay) % 2 == 0, 0.948, 1.062) \
        * (1.0 + 0.02 * rng.standard_normal(n_lay))
    vp[ch_k + 1] *= 0.87        # channel fill: soft sand
    vp[5] *= 1.09               # hard calcareous shallow marker
    i_res = n_sh + 1 + 5        # the bed that subcrops at the crest
    vp[i_res] *= 0.88           # the reservoir sand: soft, bright
    vp[n_sh + 1 + 12] *= 1.13   # carbonate stringer marker
    vp = np.clip(vp, 1580.0, 4950.0)
    ai_lay = vp * 310.0 * vp ** 0.25          # Gardner density
    idx = (zgrid[:, None, None] >= tops[None, :, :]).sum(axis=1)
    het = rng.normal(0, 1, (nz, nx))          # subtle heterogeneity
    het = np.apply_along_axis(lambda c: np.convolve(c, np.ones(5)/5, 'same'), 0, het)
    het = np.apply_along_axis(lambda c: np.convolve(c, np.ones(9)/9, 'same'), 1, het)
    ai = ai_lay[idx] * (1.0 + 0.008 * het / (het.std() + 1e-12))
    return dict(x=x, z=zgrid, xn=xn, ai=ai, vp=vp[idx], idx=idx, tops=tops,
                unconf=unconf, dx=dx, dz=dz, i_res=i_res)

def to_time(m, dt=0.002):
    # per-trace depth-to-time conversion of the impedance grid
    twt = 2.0 * m['dz'] * np.cumsum(1.0 / m['vp'], axis=0)
    tgrid = np.arange(0.0, np.percentile(twt[-1], 90), dt)
    ai_t = np.empty((tgrid.size, m['ai'].shape[1]))
    for ix in range(m['ai'].shape[1]):
        ai_t[:, ix] = np.interp(tgrid, twt[:, ix], m['ai'][:, ix])
    return tgrid, ai_t

def seismic_image(ai_t, tgrid, dt=0.002, snr=5.0, seed=11):
    # the processed migrated-image idealization: reflectivity, a wavelet
    # whose frequency falls gently with time (residual absorption after
    # Q compensation), then band-limited noise at survey-quality S/N
    rng = np.random.default_rng(seed)
    rc = np.zeros_like(ai_t)
    rc[1:, :] = (ai_t[1:, :] - ai_t[:-1, :]) / (ai_t[1:, :] + ai_t[:-1, :])
    secs = [np.apply_along_axis(
        lambda c: np.convolve(c, ricker(f0, dt, 91), 'same'), 0, rc)
        for f0 in (34.0, 27.0, 21.0)]
    t = tgrid[:, None]
    w_sh = np.clip((0.55 - t) / 0.35, 0, 1)
    w_dp = np.clip((t - 0.75) / 0.35, 0, 1)
    sec = w_sh*secs[0] + (1 - w_sh - w_dp)*secs[1] + w_dp*secs[2]
    noise = rng.normal(0, 1, sec.shape)
    noise = np.apply_along_axis(
        lambda c: np.convolve(c, ricker(27.0, dt, 91), 'same'), 0, noise)
    noise = np.apply_along_axis(
        lambda c: np.convolve(c, np.ones(3)/3, 'same'), 1, noise)
    noise *= np.sqrt((sec**2).mean()) / (snr * np.sqrt((noise**2).mean()) + 1e-12)
    return sec + noise


m = build_field_model(seed=SEED)
tg, ai_t = to_time(m, dt=DT_S)
section = seismic_image(ai_t, tg, dt=DT_S, snr=SNR)

# honesty checks: the numbers a reviewer would ask for
rc = np.zeros_like(ai_t)
rc[1:, :] = (ai_t[1:, :] - ai_t[:-1, :]) / (ai_t[1:, :] + ai_t[:-1, :])
print('Vp %.0f-%.0f m/s | reflection coefficients p50 %.3f p99 %.3f'
      % (m['vp'].min(), m['vp'].max(),
         np.median(np.abs(rc[np.abs(rc) > 1e-6])),
         np.percentile(np.abs(rc[np.abs(rc) > 1e-6]), 99)))

xkm = m['x'] / 1000.0
zkm = m['z'] / 1000.0

# --- figure 1: the geology, assembled and annotated -------------------
fig, ax = plt.subplots(figsize=(9.4, 5.0))
ax.imshow(m['ai'] / 1e6, cmap='viridis', aspect='auto',
          extent=[xkm[0], xkm[-1], zkm[-1], 0], alpha=0.92)
for k in range(m['tops'].shape[0]):
    ax.plot(xkm, m['tops'][k] / 1000.0, color='k', lw=0.3, alpha=0.35)
ax.plot(xkm, m['unconf'] / 1000.0, color='gold', lw=2.0)
res_top = m['tops'][m['i_res'] - 1] / 1000.0
ax.plot(xkm, res_top, color='crimson', lw=1.4)
bb = dict(facecolor='white', alpha=0.8, edgecolor='none', pad=1.5)
ax.annotate('angular unconformity', xy=(0.62, m['unconf'][50] / 1000.0),
            xytext=(0.35, 0.20), color='k', fontsize=9, bbox=bb,
            arrowprops=dict(arrowstyle='-', lw=0.8))
ax.annotate('reservoir sand subcrops at the crest',
            xy=(1.95, res_top[156] + 0.012), xytext=(2.6, 0.32),
            color='k', fontsize=9, bbox=bb, arrowprops=dict(arrowstyle='-', lw=0.8))
ax.annotate('channel', xy=(2.96, m['tops'][2][237] / 1000.0 + 0.05),
            xytext=(3.45, 0.10), color='k', fontsize=9, bbox=bb,
            arrowprops=dict(arrowstyle='-', lw=0.8))
ax.annotate('growth fault', xy=(2.75, 0.95), xytext=(3.6, 0.72),
            color='k', fontsize=9, bbox=bb, arrowprops=dict(arrowstyle='-', lw=0.8))
ax.set_ylim(zkm[-1], 0)
ax.set_title('the geology: folded, faulted, truncated, draped')
ax.set_xlabel('distance (km)'); ax.set_ylabel('depth (km)')
plt.tight_layout(); plt.show()

# --- figure 2: geology in, seismic out (the hero) ---------------------
fig, ax = plt.subplots(1, 2, figsize=(12.8, 4.7))
ax[0].imshow(m['ai'] / 1e6, cmap='viridis', aspect='auto',
             extent=[xkm[0], xkm[-1], zkm[-1], 0])
ax[0].set_title('the earth model: impedance in depth')
ax[0].set_xlabel('distance (km)'); ax[0].set_ylabel('depth (km)')
c = np.percentile(np.abs(section), 99)
ax[1].imshow(section, cmap='gray_r', vmin=-c, vmax=c, aspect='auto',
             extent=[xkm[0], xkm[-1], tg[-1], 0])
ax[1].set_title('the seismic image: S/N %.0f, migrated-image idealization' % SNR)
ax[1].set_xlabel('distance (km)'); ax[1].set_ylabel('two-way time (s)')
plt.tight_layout(); plt.show()

## Where to go next

- The course: https://ogbonlab.com/textbooks/synthetic-seismic-modeling
- More notebooks: https://github.com/mosesfalowo/ogbonlab-notebooks
- Scale up: Devito, SPECFEM3D, Madagascar (course Part 12.3)